## HyWett Predictive Tool

### Hydrogen Wettability Estimation via Contact Angle Using Bayesian-optimized Extremely Randomized Trees 

In [1]:
import gradio as gr
import joblib
import pandas as pd

# Model we developed
model = joblib.load("ET.joblib")

# Range of dataset used for model development
VALID_RANGES = {
    "Pressure": (3.45, 20.68),                # MPa
    "Temperature": (303.15, 343.15),          # K
    "Salinity": (0.34, 3.42),                 # mol/kg
    "xH2": (20.0, 80.0),                      # mol%
    "xN2": (5.0, 70.0),                       # mol%
    "xCH4": (5.0, 70.0),                      # mol%
    "Density difference": (835.61, 1087.31)  # kg/m^3
}

target_sum = 95.0

# Prediction function
def predict_CA(
    pressure,
    temperature,
    salinity,
    xH2,
    xN2,
    xCH4,
    density_difference
):

    values = {
        "Pressure": pressure,
        "Temperature": temperature,
        "Salinity": salinity,
        "xH2": xH2,
        "xN2": xN2,
        "xCH4": xCH4,
        "Density difference": density_difference
    }

    warnings = []

    # Validating the range provided by user
    for feature, value in values.items():
        low, high = VALID_RANGES[feature]

        if value < low or value > high:
            warnings.append(
                f"- {feature}: {value:.2f} is outside the training range "
                f"({low:.2f}–{high:.2f})"
            )

    # Validating the gas composition
    total_gas = xH2 + xN2 + xCH4

    if abs(total_gas - target_sum) > 0.1:
        warnings.append(
            f" - Gas composition sum = {total_gas:.2f}% "
            f"(expected approximately {target_sum:.0f}%)"
        )

    # Contact angle prediction by the developed model
    X = pd.DataFrame({
        "Pressure": [pressure],
        "Temperature": [temperature],
        "Salinity": [salinity],
        "xH2": [xH2],
        "xN2": [xN2],
        "xCH4": [xCH4],
        "Density difference": [density_difference]
    })

    prediction = round(model.predict(X)[0], 2)

    # Status message
    if warnings:
        status = (
            "WARNING: One or more inputs are outside the model training domain.\n\n"
            + "\n".join(warnings)
            + "\n\nPrediction was generated, but reliability may be reduced."
        )
    else:
        status = (
            "Great! All inputs are within the model training domain.\n"
            "Prediction is being made within the range of the training data."
        )

    return prediction, status


# Gradio App
with gr.Blocks(title="Contact Angle Prediction Tool") as demo:

    gr.Markdown("<h1 style='text-align: center;'>HyWett Predictive Tool</h1>")

    with gr.Row():
        # Left panel: Model Information
        with gr.Column(scale=1):

            gr.Markdown("""
## Model information

### Purpose

This model predicts **contact angle (&theta;)** from:

- Pressure (MPa)
- Temperature (K)
- Salinity (mol/kg)
- Hydrogen mole fraction (mol%)
- Nitrogen mole fraction (mol%)
- Methane mole fraction (mol%)
- Density difference (kg/m<sup>3</sup>)

### Input ranges

| Variable | Range |
|----------|--------|
| Pressure (MPa) | 3.45–20.68 |
| Temperature (K) | 303.15–343.15 |
| Salinity (mol/kg) | 0.34–3.42 |
| Hydrogen mole fraction (mol%) | 20–80 |
| Nitrogen mole fraction (mol%) | 5–70 |
| Methane mole fraction (mol%) | 5–70 |
| Density difference (kg/m<sup>3</sup>) | 836–1087 |

### Model performance

| Dataset | R<sup>2</sup> | MAE (<sup>o</sup>) | RMSE (<sup>o</sup>) |
|----------|------:|------:|------:|
| Train | 0.936 | 0.597 | 0.967 |
| Test | 0.751 | 1.236 | 1.893 | 
| All | 0.881 | 0.788 | 1.315 |


### Metric definitions

- **R<sup>2</sup>**: Coefficient of determination
- **MAE**: Mean absolute error
- **RMSE**: Root mean squared error

### Notes

- Gas composition sum should equal **95 mol%**
- Predictions outside the training ranges may be less reliable.

""")       
        
        # Right: Prediction tool
        with gr.Column(scale=2):

            gr.Markdown("### Enter input parameters here")

            pressure = gr.Number(
                label="Pressure (MPa)",
                info="Training range: 3.45–20.68"
            )

            temperature = gr.Number(
                label="Temperature (K)",
                info="Training range: 303.15–343.15"
            )

            salinity = gr.Number(
                label="Salinity (mol/kg)",
                info="Training range: 0.34–3.42"
            )

            xH2 = gr.Number(
                label="Hydrogen mole fraction (mol%)",
                info="Training range: 20–80"
            )

            xN2 = gr.Number(
                label="Nitrogen mole fraction (mol%)",
                info="Training range: 5–70"
            )

            xCH4 = gr.Number(
                label="Methane mole fraction (mol%)",
                info="Training range: 5–70"
            )

            density_difference = gr.Number(
                label="Density difference (kg/m\u00B3)",
                info="Training range: 835.61–1087.31"
            )

            predict_button = gr.Button(
                "Predict contact angle",
                variant="secondary"
            )

            prediction_output = gr.Number(
                label="Predicted contact angle (\u00B0)"
            )

            validity_output = gr.Textbox(
                label="Model validity check",
                lines=8
            )

            predict_button.click(
                fn=predict_CA,
                inputs=[
                    pressure, 
                    temperature,
                    salinity,
                    xH2,
                    xN2,
                    xCH4,
                    density_difference],
                outputs=[
                    prediction_output,
                    validity_output]
            )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
